# Pakistani Social Media Content Detective: real YouTube pilot

Official Jazz, Zong, and Ufone channel snapshots. The collector uses uploads playlists,
public video metadata, and capped newest top-level comments. Read the actual collection
report for counts/errors; no Instagram data or viewer demographics are implied.

The common window was extended to October 2025–September 2026 with user approval after
the April–September window contained only seven eligible Zong videos. Post ages still
differ across brands. NLP outputs are exploratory, unevaluated cue labels, not gold truth.


In [1]:
from pathlib import Path
import sys, json
import pandas as pd
import plotly.express as px
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.analysis import prepare, feasibility
from src.loader import Manifest, read_csv
from src.metrics import choose_metric, source_coverage, summarize
from src.content_nlp import annotate_comments
from src.reporting import markdown_report
from src.dataset_validation import validate_exports
folder = ROOT / 'data/processed'
receipt = json.loads((folder / 'youtube_collection_report.json').read_text())
raw = read_csv((folder / 'youtube_posts.csv').read_bytes())
comments = annotate_comments(pd.read_csv(folder / 'youtube_comments.csv', dtype=str, keep_default_na=False))
manifest = Manifest(file_name='youtube_posts.csv', source_name='Verified telecom YouTube pilot', source_type='official_api',
                    status='real', source_timezone='UTC', geographic_relevance='Pakistan-relevant brands, not verified Pakistani viewers',
                    notes=receipt['sampling_method'] + '\n' + receipt['limitations'])
posts, quality = prepare(raw, manifest)
audit = validate_exports(raw, comments)
assert audit['valid'], audit['errors']
metric = choose_metric(posts).name
print('Real posts:', len(posts), '| sampled public comments:', len(comments), '| primary metric:', metric)
display(pd.DataFrame(receipt['brands']))


Real posts: 30 | sampled public comments: 279 | primary metric: visible_per_1000_views


,brand,requested,posts_collected,comments_collected,status,eligible_playlist_ids,enumeration_complete,unavailable_video_ids
0,Jazz,10,10,31,complete,343,True,0
1,Zong,10,10,140,complete,41,True,0
2,Ufone,10,10,108,complete,65,True,0


In [2]:
display(source_coverage(posts))
display(feasibility(posts, metric, manifest))
print('API errors:', len(receipt['api_errors']))
print('Latest invocation network calls:', receipt['request_attempts'], '| cached responses:', receipt['cached_responses_used'])
print('Post-age ranges by brand:')
display(posts.groupby('brand').age_days_at_snapshot.agg(['min', 'median', 'max']))


,platform,brand,posts,likes_observed,comments_observed,views_observed,shares_observed,reach_observed,impressions_observed,follower_count_observed,youngest_days,oldest_days
0,YouTube,Jazz,10,10,10,10,0,0,0,10,17.739293,46.957302
1,YouTube,Ufone,10,10,10,10,0,0,0,10,49.708894,169.833604
2,YouTube,Zong,10,10,10,10,0,0,0,10,27.248864,207.875531


,question,support,explanation
0,Which platform performs better?,Partial,Requires at least two platforms with valid mea...
1,Which content type gets more engagement?,Partial,Content type and valid measurements required; ...
2,What topics and hashtags are popular?,Partial,Observed caption/tag/topic usage only; frequen...
3,What sentiment is associated with engagement?,Unavailable,Caption sentiment only. Supplied labels/baseli...
4,What posting time/day appears better?,Supported,Known timezone required for Karachi weekday/ho...
5,Which Pakistani audience segment seems more re...,Unavailable,"Requires documented segment evidence, declared..."
6,What should a Pakistani brand post next?,Partial,Descriptive experiments only; no causal or pre...


API errors: 0
Latest invocation network calls: 0 | cached responses: 115
Post-age ranges by brand:


,min,median,max
brand,,,
Jazz,17.739293,37.868217,46.957302
Ufone,49.708894,126.452655,169.833604
Zong,27.248864,129.751063,207.875531


In [3]:
display(summarize(posts, 'brand', metric))
px.box(posts, x='brand', y=metric, points='all', hover_data=['post_id', 'age_days_at_snapshot']).show()


,brand,total_posts,n,coverage_pct,mean,median,small_sample
1,Ufone,10,10,100.0,25.217170,28.531242,False
2,Zong,10,10,100.0,23.627540,24.580729,False
0,Jazz,10,10,100.0,8.235211,6.800714,False


In [4]:
display(summarize(posts, 'category_suggested', metric))
display(summarize(posts, 'language_inferred', metric))
print('These are suggested content labels; human review and evaluation are pending.')
display(posts[['brand', 'source_url', 'post_datetime', 'category_suggested', 'category_evidence', 'language_inferred']])


,category_suggested,total_posts,n,coverage_pct,mean,median,small_sample
2,product / service update,13,13,100.0,20.943372,25.203252,False
3,promotional,6,6,100.0,20.709543,24.574160,False
1,other / uncertain,10,10,100.0,16.790869,15.696791,False
0,customer support,1,1,100.0,6.369427,6.369427,True


,language_inferred,total_posts,n,coverage_pct,mean,median,small_sample
0,English candidate,18,18,100.0,25.954565,26.893404,False
1,Roman Urdu candidate,10,10,100.0,10.344250,6.800714,False
2,unknown,2,2,100.0,0.087269,0.087269,True


These are suggested content labels; human review and evaluation are pending.


,brand,source_url,post_datetime,category_suggested,category_evidence,language_inferred
0,Jazz,https://www.youtube.com/watch?v=EbqqtZCq9cc,2026-09-30T13:17:06Z,other / uncertain,,English candidate
1,Jazz,https://www.youtube.com/watch?v=5NgaeXoahLc,2026-09-29T15:40:26Z,product / service update,5g,Roman Urdu candidate
2,Jazz,https://www.youtube.com/watch?v=PN8iALpJbUU,2026-09-17T16:13:27Z,other / uncertain,,Roman Urdu candidate
3,Jazz,https://www.youtube.com/watch?v=sC5bU-CUU08,2026-09-15T14:49:01Z,product / service update,5g,Roman Urdu candidate
4,Jazz,https://www.youtube.com/watch?v=7aEIqOKZ_DY,2026-09-10T10:11:31Z,other / uncertain,customer support: complaint; educational: how ...,Roman Urdu candidate
5,Jazz,https://www.youtube.com/watch?v=Psloe1ZOPck,2026-09-10T10:11:23Z,other / uncertain,customer support: complaint; educational: how ...,Roman Urdu candidate
6,Jazz,https://www.youtube.com/watch?v=F64i92dX29Q,2026-09-08T10:25:52Z,customer support,complaint,Roman Urdu candidate
7,Jazz,https://www.youtube.com/watch?v=iuDxvRoobyM,2026-09-07T15:02:16Z,product / service update,"5g, network",Roman Urdu candidate
8,Jazz,https://www.youtube.com/watch?v=SjCLkMSJk8E,2026-09-01T13:56:29Z,promotional,package,Roman Urdu candidate
9,Jazz,https://www.youtube.com/watch?v=6q185TDWw6U,2026-09-01T08:03:10Z,product / service update,5g,Roman Urdu candidate


In [5]:
sentiment_counts = comments.groupby(['brand', 'sentiment_label']).size().reset_index(name='sampled_comments')
display(sentiment_counts)
px.bar(sentiment_counts, x='brand', y='sampled_comments', color='sentiment_label', barmode='group').show()
print('Cue-assessed:', comments.sentiment_label.ne('unassessed').sum(), '/', len(comments))
print('No human labels supplied: no accuracy, macro-F1, or calibrated confidence is claimed.')


,brand,sentiment_label,sampled_comments
0,Jazz,negative,1
1,Jazz,positive,1
2,Jazz,unassessed,29
3,Ufone,negative,6
4,Ufone,positive,3
5,Ufone,unassessed,99
6,Zong,negative,2
7,Zong,positive,10
8,Zong,unassessed,128


Cue-assessed: 23 / 279
No human labels supplied: no accuracy, macro-F1, or calibrated confidence is claimed.


In [6]:
report = markdown_report(posts, quality, metric, manifest, comments=comments)
(folder / 'youtube_findings.md').write_text(report, encoding='utf-8')
print('Saved evidence report:', folder / 'youtube_findings.md')


Saved evidence report: /Users/adnan/Documents/ChatGPT/scraper tool/data/processed/youtube_findings.md
